<a href="https://colab.research.google.com/github/YAN-JINGHAO/TorchCode/blob/main/templates/06_multihead_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/06_multihead_attention.ipynb)

# 🔴 Hard: Multi-Head Attention

Implement **Multi-Head Attention** from scratch — the core building block of the Transformer.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q,\; K W_i^K,\; V W_i^V)$$

### Signature
```python
class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, Q, K, V) -> torch.Tensor: ...
```

### Requirements
- Use `nn.Linear(d_model, d_model)` for `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`
- `d_k = d_model // num_heads` per head
- `forward(Q, K, V)`: Q is `(B, seq_q, d_model)`, K/V are `(B, seq_k, d_model)`
- Must support **cross-attention** (`seq_q != seq_k`)
- Do **NOT** use `torch.nn.MultiheadAttention`
- You **may** use `torch.softmax` and `torch.matmul`

### Steps
1. Project: `q = self.W_q(Q)`, `k = self.W_k(K)`, `v = self.W_v(V)`
2. Reshape to `(B, num_heads, seq, d_k)`
3. Scaled dot-product attention per head
4. Concat heads → `(B, seq_q, d_model)`
5. Output projection: `self.W_o(concat)`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.9 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [12]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int):
        # Initialize W_q, W_k, W_v, W_o
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = torch.nn.Linear(d_model, d_model)
        self.W_k = torch.nn.Linear(d_model, d_model)
        self.W_v = torch.nn.Linear(d_model, d_model)
        self.W_o = torch.nn.Linear(d_model, d_model)

    def forward(self, Q, K, V):
        # Implement multi-head attention
        B, seq_q, _ = Q.size()
        seq_k = K.size(1)
        q = self.W_q(Q).reshape(B, seq_q, -1, self.d_k).transpose(1, 2)
        k = self.W_k(K).reshape(B, seq_k, -1, self.d_k).transpose(1, 2)
        v = self.W_v(V).reshape(B, seq_k, -1, self.d_k).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(2, 3)) / math.sqrt(self.d_k) # k.transpose(-2, -1)
        weights = torch.softmax(scores, dim=-1)
        O = torch.matmul(weights, v).transpose(1, 2).reshape(B, seq_q, -1)
        return self.W_o(O)


In [7]:
Q = torch.randn(1, 3, 32)
print(Q)
Q_re = torch.reshape(Q, (1, 3, 4, 8)).transpose(1, 2)
# print(Q)
print(Q_re)

tensor([[[-0.4381,  2.5384,  0.4328, -1.3571, -0.4314,  1.4922, -0.4685,
          -0.1422, -0.3681, -0.4367, -0.1030, -0.5629, -3.3252, -0.9982,
          -0.9606, -1.4633, -0.1027,  1.2920, -0.4413, -0.4657, -0.4010,
           0.2739, -1.8310, -0.2107,  1.1660, -0.8307,  0.0181,  0.3181,
           0.6938,  0.2940, -0.9989, -0.7286],
         [ 0.8089,  0.1314,  0.2491, -0.2686, -0.7965, -1.5231,  0.7699,
           0.7869, -0.4671,  1.5809,  0.5092, -0.8806,  1.2485, -1.3804,
          -0.8955,  1.1219, -1.6490,  1.2287, -0.3036, -0.6746, -0.5031,
          -0.1335, -1.1188, -1.2914, -0.5119, -0.4514, -0.4203, -0.9234,
          -0.0354,  0.7550, -1.3802, -1.9041],
         [-0.4965,  0.4263, -1.4987, -0.3378, -0.9326,  0.4664,  0.2588,
           1.6036,  1.0020,  0.4567, -0.6826, -0.5923, -0.2725, -2.7405,
           2.2126,  0.7997, -1.0574, -0.8221, -0.5173, -0.9251, -0.1484,
          -0.5200,  0.3066, -0.6861,  1.2369, -1.1059, -0.9362,  0.9500,
           1.2307, -0.7759, -0

In [13]:
# 🧪 Debug
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, num_heads=4)
print("W_q type:", type(mha.W_q))          # should be nn.Linear
print("W_q.weight shape:", mha.W_q.weight.shape)  # (32, 32)

x = torch.randn(2, 6, 32)
out = mha.forward(x, x, x)
print("Output shape:", out.shape)          # (2, 6, 32)

# Cross-attention
Q = torch.randn(1, 3, 32)
K = torch.randn(1, 7, 32)
V = torch.randn(1, 7, 32)
out2 = mha.forward(Q, K, V)
print("Cross-attn shape:", out2.shape)     # (1, 3, 32)

W_q type: <class 'torch.nn.modules.linear.Linear'>
W_q.weight shape: torch.Size([32, 32])
Output shape: torch.Size([2, 6, 32])
Cross-attn shape: torch.Size([1, 3, 32])


In [14]:
# ✅ SUBMIT
from torch_judge import check
check("mha")


🧪 Testing: Multi-Head Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/6] Output shape (2.9ms)
  ✅ [2/6] Uses nn.Linear with correct shapes (1.0ms)
  ✅ [3/6] Numerical correctness vs reference (13.4ms)
  ✅ [4/6] Gradient flow (21.2ms)
  ✅ [5/6] Cross-attention (seq_q != seq_k) (1.1ms)
  ✅ [6/6] Different heads give different outputs (2.0ms)
──────────────────────────────────────────────────
  🎉 All 6 tests passed! (41.6ms total)
  Progress saved. Run status() to see your dashboard.

